In [ ]:
# Cell 1: Import
import sys
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt
from skimage import data
from skimage.metrics import structural_similarity

from noise import add_gaussian_noise
from filters import gaussian_filter
from metrics import psnr, ssim
from nlm import nlm_denoise_fast
# Prefer a real bm3d implementation if available, otherwise fall back to a compatible denoiser
try:
    import bm3d as _bm3d_pkg

    def bm3d_denoise(noisy, sigma_psd):
        # bm3d typically expects floats in [0,1]
        arr = noisy.astype(np.float32) / 255.0
        den = _bm3d_pkg.bm3d(arr, sigma_psd / 255.0)
        return np.clip(den * 255.0, 0, 255).astype(np.uint8)

except ModuleNotFoundError:
    # Fallback: use scikit-image's NL-means as a stand-in for BM3D
    from skimage.restoration import denoise_nl_means

    def bm3d_denoise(noisy, sigma_psd):
        arr = noisy.astype(np.float32) / 255.0
        h = sigma_psd / 255.0
        den = denoise_nl_means(arr, h=h, fast_mode=True, patch_size=5, patch_distance=6, multichannel=False)
        return np.clip(den * 255.0, 0, 255).astype(np.uint8)

img = data.camera().astype(np.uint8)
print(f"Ảnh: {img.shape}, range [{img.min()}, {img.max()}]")

ModuleNotFoundError: No module named 'bm3d'

In [ ]:
# Cell 2: Test NLM với các mức sigma khác nhau
sigmas = [15, 25, 50]
fig, axes = plt.subplots(len(sigmas), 4, figsize=(16, 4*len(sigmas)))

for row, sigma in enumerate(sigmas):
    noisy    = add_gaussian_noise(img, sigma=sigma)
    gauss    = gaussian_filter(noisy, kernel_size=5, sigma=1.5)
    nlm_out  = nlm_denoise_fast(noisy, patch_size=5, search_size=21, h=sigma)
    bm3d_out = bm3d_denoise(noisy, sigma_psd=sigma)
    
    results = [
        ("Noisy",          noisy,    psnr(img, noisy)),
        ("Gaussian filter",gauss,    psnr(img, gauss)),
        ("NLM",            nlm_out,  psnr(img, nlm_out)),
        ("BM3D",           bm3d_out, psnr(img, bm3d_out)),
    ]
    
    for col, (title, result, p) in enumerate(results):
        axes[row, col].imshow(result, cmap='gray')
        axes[row, col].set_title(f"{title}\nσ={sigma}, PSNR={p:.2f} dB", fontsize=10)
        axes[row, col].axis('off')

plt.suptitle("So sánh 4 phương pháp trên 3 mức noise", fontsize=13)
plt.tight_layout()
plt.savefig('../results/week2_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 3: Visualize "non-local" nature của NLM
# Đây là điểm khác biệt lớn nhất — hiểu được cái này = hiểu NLM thực sự

from skimage.restoration import denoise_nl_means, estimate_sigma

# Chọn 1 pixel và visualize weight của nó với tất cả pixel khác
img_small = img[100:200, 100:200]   # crop nhỏ để tính nhanh
noisy_small = add_gaussian_noise(img_small, sigma=25)

H, W = img_small.shape
query_i, query_j = 30, 30   # pixel muốn xem weight map
patch_size = 5
half_p = patch_size // 2
h = 25

pad = half_p
padded = np.pad(noisy_small.astype(float), pad, mode='reflect')

# Patch của query pixel
pi, pj = query_i + pad, query_j + pad
patch_query = padded[pi-half_p:pi+half_p+1, pj-half_p:pj+half_p+1]

# Tính weight map
weight_map = np.zeros((H, W))
for i in range(H):
    for j in range(W):
        p = padded[i+pad-half_p:i+pad+half_p+1, j+pad-half_p:j+pad+half_p+1]
        dist_sq = np.sum((patch_query - p)**2)
        weight_map[i, j] = np.exp(-dist_sq / h**2)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(noisy_small, cmap='gray')
axes[0].plot(query_j, query_i, 'r+', markersize=15, markeredgewidth=2)
axes[0].set_title(f"Ảnh nhiễu\n(dấu + = query pixel tại [{query_i},{query_j}])")

axes[1].imshow(weight_map, cmap='hot')
axes[1].plot(query_j, query_i, 'b+', markersize=15, markeredgewidth=2)
axes[1].set_title("Weight map\n(trắng = weight cao, đen = weight thấp)")
plt.colorbar(axes[1].images[0], ax=axes[1])

axes[2].imshow(img_small, cmap='gray')
axes[2].plot(query_j, query_i, 'r+', markersize=15, markeredgewidth=2)
axes[2].set_title("Ảnh gốc\n(so sánh vùng sáng với weight map)")

for ax in axes: ax.axis('off')
plt.suptitle("NLM Weight Map: pixel nào ảnh hưởng đến pixel query?", fontsize=12)
plt.tight_layout()
plt.savefig('../results/week2_nlm_weights.png', dpi=150, bbox_inches='tight')
plt.show()

# Quan sát: weight cao (màu trắng) xuất hiện ở các vùng
# có texture GIỐNG NHAU với query pixel, dù ở xa trong ảnh

In [ ]:
# Cell 4: Benchmark table chính thức
import pandas as pd

methods = {
    'Gaussian filter' : lambda n, s: gaussian_filter(n, 5, 1.5),
    'NLM'             : lambda n, s: nlm_denoise_fast(n, h=s),
    'BM3D'            : lambda n, s: bm3d_denoise(n, sigma_psd=s),
}

sigmas = [15, 25, 50]
records = []

for sigma in sigmas:
    noisy = add_gaussian_noise(img, sigma=sigma)
    for name, fn in methods.items():
        result = fn(noisy, sigma)
        records.append({
            'Method': name,
            'Sigma' : sigma,
            'PSNR'  : round(psnr(img, result),  2),
            'SSIM'  : round(ssim(img, result),  4),
        })

df = pd.DataFrame(records)
print(df.pivot_table(index='Method', columns='Sigma', values=['PSNR','SSIM']))
df.to_csv('../results/week2_benchmark.csv', index=False)